# Send registration confirmation emails

Reads new rows from the `event_sign_up` silver table and sends one email
per registration through a custom `gmail` Spark Python data source defined
inline below.

The data source class is defined **in this notebook** (rather than
imported from `../scripts/send_gmail.py`) for two reasons:

1. Notebook code runs in `__main__`, which makes cloudpickle ship the
   class to Spark executors **by value**. No `register_pickle_by_value`
   gymnastics needed.
2. No filesystem/sys.path/`%run` setup — the notebook is self-contained
   and works on any cluster runtime that supports the Python data source
   API.

## One-time setup: store the Gmail App Password as a Databricks secret

The Gmail App Password is read from a Databricks secret scope, never from
a widget or a hard-coded string. Create the scope and put the password in
once from your laptop:

```bash
databricks secrets create-scope cdp_demo -p FEVM
databricks secrets put-secret cdp_demo gmail_app_password -p FEVM
# paste the 16-character app password when prompted, then Ctrl-D
```

The widgets below let you point at a different scope / key without
editing the notebook.

In [ ]:
dbutils.widgets.text("secret_scope", "cdp_demo", "Secret scope")
dbutils.widgets.text("secret_key", "gmail_app_password", "Secret key (Gmail app password)")

GMAIL_ADDRESS = "nikolaos.servos@gmail.com"
GMAIL_APP_PASSWORD = dbutils.secrets.get(
    scope=dbutils.widgets.get("secret_scope"),
    key=dbutils.widgets.get("secret_key"),
)

print({
    "gmail_address": GMAIL_ADDRESS,
    "app_password_loaded": bool(GMAIL_APP_PASSWORD),
})

In [ ]:
import mimetypes
import os
import smtplib
from email.message import EmailMessage
from typing import Iterable, Optional

from pyspark.sql.datasource import (
    DataSource,
    DataSourceStreamWriter,
    WriterCommitMessage,
)
from pyspark.sql.types import StructType


def send_gmail_email(
    to_addresses,
    subject: str,
    body_text: str,
    *,
    gmail_address: Optional[str] = None,
    app_password: Optional[str] = None,
    cc_addresses: Optional[Iterable[str]] = None,
    bcc_addresses: Optional[Iterable[str]] = None,
    body_html: Optional[str] = None,
    attachment_paths: Optional[Iterable[str]] = None,
):
    """Send an email via Gmail SMTPS.

    Prefer reading `gmail_address` and `app_password` from Databricks secrets.
    For Gmail, use an App Password (2-Step Verification must be enabled).
    """
    if isinstance(to_addresses, str):
        to_addresses = [to_addresses]
    else:
        to_addresses = list(to_addresses)

    cc_addresses = [] if cc_addresses is None else list(cc_addresses)
    bcc_addresses = [] if bcc_addresses is None else list(bcc_addresses)
    attachment_paths = [] if attachment_paths is None else list(attachment_paths)

    if not to_addresses:
        raise ValueError("Provide at least one recipient in to_addresses.")
    if not gmail_address:
        raise ValueError("gmail_address is required.")
    if not app_password:
        raise ValueError("app_password is required (Gmail App Password).")

    message = EmailMessage()
    message["Subject"] = subject
    message["From"] = gmail_address
    message["To"] = ", ".join(to_addresses)
    if cc_addresses:
        message["Cc"] = ", ".join(cc_addresses)
    message.set_content(body_text)
    if body_html:
        message.add_alternative(body_html, subtype="html")

    for path in attachment_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Attachment not found: {path}")
        mime_type, _ = mimetypes.guess_type(path)
        if mime_type:
            maintype, subtype = mime_type.split("/", 1)
        else:
            maintype, subtype = "application", "octet-stream"
        with open(path, "rb") as f:
            message.add_attachment(
                f.read(),
                maintype=maintype,
                subtype=subtype,
                filename=os.path.basename(path),
            )

    all_recipients = to_addresses + cc_addresses + bcc_addresses
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(gmail_address, app_password)
        smtp.send_message(message, to_addrs=all_recipients)

    return {
        "status": "sent",
        "from": gmail_address,
        "to": to_addresses,
        "cc": cc_addresses,
        "bcc_count": len(bcc_addresses),
        "attachment_count": len(attachment_paths),
        "subject": subject,
    }


class EmailWriterCommitMessage(WriterCommitMessage):
    def __init__(self, partition_id: int, sent_count: int):
        self.partition_id = partition_id
        self.sent_count = sent_count


class GmailStreamWriter(DataSourceStreamWriter):
    def __init__(self, options):
        self.options = options
        self.gmail_address = self.options.get("gmailAddress")
        self.app_password = self.options.get("appPassword")
        self.target_recipient_col = self.options.get("targetRecipientCol")
        self.subject_col = self.options.get("subjectCol")
        self.body_col = self.options.get("bodyCol")

        if not self.gmail_address:
            raise ValueError("The option 'gmailAddress' is required.")
        if not self.app_password:
            raise ValueError("The option 'appPassword' is required.")
        if not self.target_recipient_col:
            raise ValueError("The option 'targetRecipientCol' is required.")
        if not self.subject_col:
            raise ValueError("The option 'subjectCol' is required.")
        if not self.body_col:
            raise ValueError("The option 'bodyCol' is required.")

    def write(self, iterator):
        from pyspark import TaskContext

        context = TaskContext.get()
        partition_id = context.partitionId() if context else 0
        sent_count = 0

        for row in iterator:
            row_dict = row.asDict(recursive=True)
            recipient = row_dict.get(self.target_recipient_col)
            subject = row_dict.get(self.subject_col)
            body_text = row_dict.get(self.body_col)

            if recipient is None:
                raise ValueError(
                    f"Column '{self.target_recipient_col}' is missing or null in an input row."
                )
            if subject is None:
                raise ValueError(
                    f"Column '{self.subject_col}' is missing or null in an input row."
                )
            if body_text is None:
                raise ValueError(
                    f"Column '{self.body_col}' is missing or null in an input row."
                )

            send_gmail_email(
                to_addresses=str(recipient),
                subject=str(subject),
                body_text=str(body_text),
                gmail_address=self.gmail_address,
                app_password=self.app_password,
            )
            sent_count += 1

        return EmailWriterCommitMessage(partition_id=partition_id, sent_count=sent_count)

    def commit(self, messages, batchId) -> None:
        total_sent = sum(m.sent_count for m in messages)
        print({"batch_id": batchId, "partitions": len(messages), "emails_sent": total_sent})

    def abort(self, messages, batchId) -> None:
        total_sent = sum(m.sent_count for m in messages)
        print({"batch_id": batchId, "status": "aborted", "emails_sent_before_abort": total_sent})


class GmailDataSource(DataSource):
    @classmethod
    def name(cls):
        return "gmail"

    def schema(self):
        return "ID long, target_recipient_col string, subject_col string, body_col string"

    def streamWriter(self, schema: StructType, overwrite: bool):
        return GmailStreamWriter(self.options)

In [ ]:
spark.dataSource.register(GmailDataSource)

source_table_name = "gtm_zerobus_demo_catalog.landing.event_sign_up"

print({"format": "gmail", "source_table_name": source_table_name})

In [ ]:
checkpoint_path = (
    "/Volumes/gtm_zerobus_demo_catalog/landing/logs/"
    "checkpoints/send_mail_create_account_man"
)

email_query = (
    spark.readStream.table(source_table_name)
         .writeStream
         .format("gmail")
         .option("checkpointLocation", checkpoint_path)
         .option("gmailAddress", GMAIL_ADDRESS)
         .option("appPassword", GMAIL_APP_PASSWORD)
         .option("targetRecipientCol", "email")
         .option("subjectCol", "subject")
         .option("bodyCol", "body")
         .trigger(availableNow=True)
         .start()
)

email_query.awaitTermination()
print({"status": email_query.status, "checkpoint_path": checkpoint_path})